# An MLP on interleaved spirals

A straight line cannot separate three interleaved spirals, but a multi-layer perceptron (MLP) with non-linear activations can. Here we use the project's `fit` helper instead of writing the loop by hand.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn

from pytorch_examples.data import make_loader, make_spirals, train_val_split
from pytorch_examples.models import MLP
from pytorch_examples.train import accuracy, fit
from pytorch_examples.utils import count_parameters, set_seed

set_seed(0)
x, y = make_spirals(n_per_class=100, n_classes=3, seed=0)
x_tr, y_tr, x_val, y_val = train_val_split(x, y, val_fraction=0.2, seed=0)
print("train:", len(x_tr), " validation:", len(x_val))

In [ ]:
plt.figure(figsize=(4, 4))
plt.scatter(x[:, 0], x[:, 1], c=y, cmap="viridis", s=12)
plt.title("Three spirals")
plt.axis("equal")
plt.tight_layout()
plt.show()

## Train

`fit` runs the training loop and records per-epoch losses plus a validation metric (here, accuracy).

In [ ]:
model = MLP(in_features=2, hidden=64, out_features=3, depth=2)
print("parameters:", count_parameters(model))

history = fit(
    model,
    make_loader(x_tr, y_tr, batch_size=32, seed=0),
    make_loader(x_val, y_val, batch_size=64, shuffle=False),
    loss_fn=nn.CrossEntropyLoss(),
    epochs=100,
    lr=1e-2,
    device="cpu",
    metric_fn=accuracy,
    verbose=True,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].plot(history.train_loss, label="train")
axes[0].plot(history.val_loss, label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()
axes[1].plot(history.val_metric)
axes[1].set_title("Validation accuracy")
axes[1].set_xlabel("epoch")
plt.tight_layout()
plt.show()

## Decision boundary

Predict the class for every point on a grid to see how the network carved up the plane.

In [ ]:
xs = torch.linspace(-1.2, 1.2, 200)
grid_x, grid_y = torch.meshgrid(xs, xs, indexing="xy")
points = torch.stack([grid_x.flatten(), grid_y.flatten()], dim=1)

model.eval()
with torch.no_grad():
    regions = model(points).argmax(dim=1).reshape(grid_x.shape)

plt.figure(figsize=(4.5, 4.5))
plt.contourf(grid_x, grid_y, regions, alpha=0.3, cmap="viridis", levels=[-0.5, 0.5, 1.5, 2.5])
plt.scatter(x[:, 0], x[:, 1], c=y, cmap="viridis", s=10, edgecolors="none")
plt.title("Learned regions")
plt.axis("equal")
plt.tight_layout()
plt.show()

In [ ]:
final_accuracy = history.val_metric[-1]
print(f"validation accuracy: {final_accuracy:.2f}  (chance is 0.33)")
assert final_accuracy > 0.6